In [ ]:
# tensorflow pytorch를 중복으로 사용시 OpenMP 라이브러리 실행 허용
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"


In [2]:
from ultralytics import YOLO

model = YOLO('./pkl/yolov8s.pt')
model.names

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

In [3]:
# 폴더에서 opencv를 이용한 이미지 파일 읽기
import cv2

image_path = './image/riverside.JPG'
img = cv2.imread(image_path)
img.shape

(472, 700, 3)

In [4]:
# TF (NWHC) => (1, 640, 448,   3)
# PT (NCHW) => (1,   3, 448, 640)
result = model(img)
result = result[0]
result


0: 448x640 9 persons, 1 car, 1 boat, 1 dog, 185.8ms
Speed: 2.1ms preprocess, 185.8ms inference, 1.7ms postprocess per image at shape (1, 3, 448, 640)


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant',

In [5]:
# 0 person, 8 boat
class_ids = result.boxes.cls
class_ids

tensor([ 0.,  2.,  0.,  0.,  8., 16.,  0.,  0.,  0.,  0.,  0.,  0.])

In [6]:
# 신뢰도
confidences = result.boxes.conf
confidences

tensor([0.8863, 0.8565, 0.8537, 0.8386, 0.8059, 0.7831, 0.7595, 0.6718, 0.6228, 0.5655, 0.5280, 0.4414])

In [7]:
class_ids = result.boxes.cls.cpu().numpy().astype(int)
class_ids

array([ 0,  2,  0,  0,  8, 16,  0,  0,  0,  0,  0,  0])

In [8]:
# cpu().numpy().타입
confidences = result.boxes.conf.cpu().numpy()
confidences

array([    0.88634,     0.85652,     0.85374,     0.83857,     0.80589,     0.78313,     0.75952,     0.67183,     0.62281,     0.56552,     0.52802,     0.44145], dtype=float32)

In [9]:
for i, cls_id in enumerate(class_ids):
    print(f"{i+1}, {model.names[cls_id]}, {confidences[i]:.2f}")

1, person, 0.89
2, car, 0.86
3, person, 0.85
4, person, 0.84
5, boat, 0.81
6, dog, 0.78
7, person, 0.76
8, person, 0.67
9, person, 0.62
10, person, 0.57
11, person, 0.53
12, person, 0.44


In [10]:
rendered_img = result.plot()

cv2.imshow("yolo", rendered_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

---

In [11]:
# 신뢰도가 80이상
result = model(img)[0]

class_ids = []      # 클래스 번호 보관용 리스트
confidences = []    # 신뢰도 보관용 리스트
name = []           # 클래스 보관용 리스트
filtered_boxes = [] # 박스 보관 리스트

# cls (클래스 아이디), conf (신뢰도 점수), xyxy (왼쪽위 xy좌표, 오른쪽아래 xy좌표)
for box in result.boxes:
    conf = float(box.conf[0])
    print(conf)

    if conf < 0.8:
        continue

    # 여기서 0.8 이상 처리
    cls = int(box.cls[0])

    # map(변경타입, [x1, x2, y1, y2]배열)
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    print(x1, y1, x2, y2)

    filtered_boxes.append((x1, y1, x2, y2, cls, conf))


0: 448x640 9 persons, 1 car, 1 boat, 1 dog, 154.9ms
Speed: 5.4ms preprocess, 154.9ms inference, 1.0ms postprocess per image at shape (1, 3, 448, 640)
0.8863399028778076
102 271 177 384
0.8565224409103394
317 189 567 276
0.8537421226501465
20 283 96 392
0.8385708332061768
463 263 569 353
0.8058911561965942
51 222 256 263
0.7831264734268188
0.7595164179801941
0.6718257069587708
0.6228057742118835
0.565523087978363
0.5280197858810425
0.4414457380771637


In [ ]:
image_path = './image/riverside.JPG'
img = cv2.imread(image_path)

for x1, y1, x2, y2, cls, conf in filtered_boxes:
    label = f"{model.names[cls]} : {conf:.2f}"

    # cv2.rectangle(이미지, (x1, y1), (x2, y2), 색상, 두께)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)

    # cv2.putText(이미지, 문자, (x,y), 폰트, 폰트크기, 색상, 두께)
    cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    # 탐지된 이미지 자르기
    # crop = img[y1:y2, x1:x2]

cv2.imshow("yolo", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [33]:
import cv2
video_path = "./video/busstop.mp4"
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    result = model(frame)[0]

    rendered_frame = result.plot()

    cv2.imshow("yolo", rendered_frame)

    # ESC 또는 q 키를 누르면 종료
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 640x384 5 persons, 1 car, 2 trucks, 1 traffic light, 1 stop sign, 111.4ms
Speed: 1.5ms preprocess, 111.4ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 6 persons, 1 car, 3 trucks, 1 traffic light, 1 stop sign, 122.7ms
Speed: 1.7ms preprocess, 122.7ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 6 persons, 1 car, 2 trucks, 1 traffic light, 1 stop sign, 1 tv, 117.8ms
Speed: 2.0ms preprocess, 117.8ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 6 persons, 1 car, 2 trucks, 1 traffic light, 1 stop sign, 1 handbag, 116.4ms
Speed: 1.5ms preprocess, 116.4ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 6 persons, 1 car, 2 trucks, 1 traffic light, 1 stop sign, 1 handbag, 116.8ms
Speed: 1.7ms preprocess, 116.8ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 7 persons, 1 car, 2 trucks, 1 traffic light, 1 stop sign, 1 handbag, 117.1ms
Spe